# ПЗ 8 — Постобработка результатов

Дедупликация текста из OCR (fuzzy matching), склейка детекций YOLO по временным окнам, анализ LLM-описаний и финальный сводный отчёт.

In [ ]:
!pip install rapidfuzz pandas matplotlib -q

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

RESULTS_DIR = '/content/drive/MyDrive/cv-frames/результаты'

print('файлы в результатах:')
for f in sorted(os.listdir(RESULTS_DIR)):
    print(f'  {f}')


## 1. Дедупликация текста (OCR)

In [ ]:
import pandas as pd
from rapidfuzz import fuzz, process

df_ocr = pd.read_csv(f'{RESULTS_DIR}/ocr_results.csv')
print(f'строк до дедупликации: {len(df_ocr)}')

def deduplicate(texts, threshold=85):
    unique = []
    for t in texts:
        if not unique:
            unique.append(t)
            continue
        best = process.extractOne(t, unique, scorer=fuzz.ratio)
        if best is None or best[1] < threshold:
            unique.append(t)
    return unique

unique = deduplicate(df_ocr['text'].dropna().tolist())
print(f'строк после дедупликации: {len(unique)}')
print('уникальные тексты:')
for t in unique[:10]:
    print(f'  {t}')

pd.DataFrame({'text': unique}).to_csv(f'{RESULTS_DIR}/ocr_dedup.csv', index=False, encoding='utf-8-sig')
print('сохранено: ocr_dedup.csv')


## 2. Склейка детекций YOLO

In [ ]:
df_det = pd.read_csv(f'{RESULTS_DIR}/yolo_detections.csv')

WINDOW = 5  # допустимый разрыв между кадрами одного события

def merge_detections(group):
    group = group.sort_values('frame_num').reset_index(drop=True)
    events = []
    start = group.iloc[0]
    prev  = start['frame_num']
    for _, row in group.iloc[1:].iterrows():
        if row['frame_num'] - prev > WINDOW:
            events.append({
                'class':       start['class'],
                'start_frame': start['frame_num'],
                'end_frame':   prev,
                'avg_conf':    round(group['conf'].mean(), 3),
            })
            start = row
        prev = row['frame_num']
    events.append({
        'class':       start['class'],
        'start_frame': start['frame_num'],
        'end_frame':   prev,
        'avg_conf':    round(group['conf'].mean(), 3),
    })
    return pd.DataFrame(events)

df_merged = (df_det
    .groupby('class', group_keys=False)
    .apply(merge_detections)
    .sort_values('start_frame')
    .reset_index(drop=True))

df_merged.to_csv(f'{RESULTS_DIR}/yolo_merged.csv', index=False)
print(f'детекций: {len(df_det)} -> событий: {len(df_merged)}')
df_merged


## 3. Анализ LLM-описаний (OpenRouter)

In [ ]:
import json
from collections import Counter

with open(f'{RESULTS_DIR}/llm_detections.json', encoding='utf-8') as f:
    llm_data = json.load(f)

print(f'обработано кадров: {len(llm_data)}')
print()

# собираем все объекты из описаний
all_objects = []
for item in llm_data:
    objects = item.get('objects', '')
    if objects and objects != 'ошибка':
        for obj in objects.split(','):
            obj = obj.strip().lower()
            if obj:
                all_objects.append(obj)

counter = Counter(all_objects)
print('топ-15 объектов из LLM:')
for obj, cnt in counter.most_common(15):
    print(f'  {obj}: {cnt}')


## 4. Визуализация результатов

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# OCR до/после
axes[0].bar(['до дедупликации', 'после дедупликации'],
            [len(df_ocr), len(unique)], color=['#e74c3c', '#2ecc71'])
axes[0].set_title('OCR: дедупликация текста')
axes[0].set_ylabel('количество строк')
for i, v in enumerate([len(df_ocr), len(unique)]):
    axes[0].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

# YOLO детекции vs события
axes[1].bar(['детекций', 'событий'],
            [len(df_det), len(df_merged)], color=['#3498db', '#9b59b6'])
axes[1].set_title('YOLO: склейка по времени')
axes[1].set_ylabel('количество')
for i, v in enumerate([len(df_det), len(df_merged)]):
    axes[1].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

# Топ объектов LLM
top_obj = counter.most_common(10)
labels = [x[0][:20] for x in top_obj]
values = [x[1] for x in top_obj]
axes[2].barh(labels[::-1], values[::-1], color='#e67e22')
axes[2].set_title('LLM: топ объектов на кадрах')
axes[2].set_xlabel('количество упоминаний')

plt.suptitle('ПЗ 8 — Постобработка результатов', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/postprocessing_summary.png', dpi=100, bbox_inches='tight')
plt.show()
print('график сохранён: postprocessing_summary.png')


## 5. Финальный отчёт

In [ ]:
import json

report = {
    'ocr': {
        'total_texts': len(df_ocr),
        'unique_texts': len(unique),
        'dedup_ratio': round(len(unique) / max(len(df_ocr), 1), 3),
    },
    'yolo': {
        'total_detections': len(df_det),
        'merged_events': len(df_merged),
        'classes': df_merged['class'].unique().tolist(),
    },
    'llm': {
        'frames_processed': len(llm_data),
        'model': 'nvidia/nemotron-nano-12b-v2-vl:free (OpenRouter)',
        'top_objects': [obj for obj, _ in counter.most_common(10)],
    }
}

with open(f'{RESULTS_DIR}/final_report.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print('=== ФИНАЛЬНЫЙ ОТЧЁТ ===')
print(f"OCR: {report['ocr']['total_texts']} строк → {report['ocr']['unique_texts']} уникальных")
print(f"YOLO: {report['yolo']['total_detections']} детекций → {report['yolo']['merged_events']} событий")
print(f"LLM: {report['llm']['frames_processed']} кадров, модель: {report['llm']['model']}")
print(f"\nсохранено: final_report.json")
